# Customer Churn Prediction using Explainable AI

## Notebook 02: Data Cleaning

## Notebook Overview

Data cleaning is an important step in the machine learning pipeline. The quality of the data directly affects the reliability of the model.

In this notebook, the original Telco Customer Churn dataset will be cleaned by handling missing values, correcting data types, removing unnecessary variables, and preventing data leakage.

The cleaned dataset will then be saved and used in the subsequent feature engineering and exploratory analysis phases.

## Objectives

The objectives of this notebook are to:

- Load the original Telco Customer Churn dataset.
- Check the dataset for duplicate records.
- Investigate missing values.
- Correct incorrect data types.
- Handle the `Total Charges` column.
- Remove irrelevant and constant columns.
- Remove potential data leakage variables.
- Verify the cleaned dataset.
- Save the cleaned dataset for future notebooks.

## Importing Required Libraries:

In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows",None)

## Load Dataset

The original Telco Customer Churn dataset is loaded for cleaning. The original dataset is kept unchanged, while a separate DataFrame is created for the cleaning process.

In [ ]:
file_path = "/content/Telco_customer_churn.xlsx"
df = pd.read_excel(file_path)
print(df.head())
print(df.shape)

   CustomerID  Count        Country       State         City  Zip Code  \
0  3668-QPYBK      1  United States  California  Los Angeles     90003   
1  9237-HQITU      1  United States  California  Los Angeles     90005   
2  9305-CDSKC      1  United States  California  Los Angeles     90006   
3  7892-POOKP      1  United States  California  Los Angeles     90010   
4  0280-XJGEX      1  United States  California  Los Angeles     90015   

                 Lat Long   Latitude   Longitude  Gender Senior Citizen  \
0  33.964131, -118.272783  33.964131 -118.272783    Male             No   
1   34.059281, -118.30742  34.059281 -118.307420  Female             No   
2  34.048013, -118.293953  34.048013 -118.293953  Female             No   
3  34.062125, -118.315709  34.062125 -118.315709  Female             No   
4  34.039224, -118.266293  34.039224 -118.266293    Male             No   

  Partner Dependents  Tenure Months Phone Service Multiple Lines  \
0      No         No              2 

## Create Working Copy

A copy of the original dataset is created so that the original data remains unchanged. All cleaning operations will be performed on this working copy.

In [ ]:
cleaned_df = df.copy()
print(cleaned_df.shape)

(7043, 33)


## Check Duplicate Records

Duplicate records can introduce bias into analysis and machine learning models. Therefore, duplicate rows are checked before further preprocessing.

In [ ]:
print(cleaned_df.duplicated().sum())

0


## Missing Values

Missing values are examined to determine which variables require special treatment.

Not all missing values represent errors. For example, `Churn Reason` is naturally unavailable for customers who did not churn.

In [ ]:
missing = cleaned_df.isnull().sum()
#print(missing)
print(missing[missing>0])

Churn Reason    5174
dtype: int64


## Churn Reason

The `Churn Reason` column contains information about why a customer left the company.

It is missing for customers who did not churn. Therefore, these missing values are not treated as ordinary data-quality errors.

Since `Churn Reason` is also known only after a customer has churned, it will not be used as a predictive feature.

In [ ]:
print(cleaned_df["Churn Reason"].isnull().sum())

5174


## Convert Total Charges to Numeric

The `Total Charges` column represents the total amount charged to each customer. However, it is currently stored as an object data type.

It will be converted to a numeric data type. Any values that cannot be converted will temporarily be represented as missing values so they can be investigated.

In [ ]:
cleaned_df["Total Charges"] = pd.to_numeric(cleaned_df["Total Charges"],errors = "coerce")
#print(cleaned_df.dtypes)
# checking newly created missing values in Total Charges column:
print(cleaned_df["Total Charges"].isnull().sum())

11


**errors = "coerce" means:**

"If you can't convert a value into a number, replace it with NaN instead of giving me an error."

In [ ]:
print(cleaned_df.loc[cleaned_df["Total Charges"].isnull(), "Tenure Months"])

2234    0
2438    0
2568    0
2667    0
2856    0
4331    0
4687    0
5104    0
5719    0
6772    0
6840    0
Name: Tenure Months, dtype: int64


## Handling Missing Values:

In [ ]:
cleaned_df["Total Charges"] = cleaned_df["Total Charges"].fillna(0)
print(cleaned_df["Total Charges"].isnull().sum())

0


## Remove Constant Columns

Columns containing only a single unique value provide no useful information for machine learning because they cannot distinguish between customers.

The `Country`,`Count` and `State` columns contain the same value for every customer and will therefore be removed.

In [ ]:
costant_columns = ["Country","State","Count"]
cleaned_df = cleaned_df.drop(columns = costant_columns)
print(cleaned_df.columns)

(7043, 30)


## Remove Identifier and Reporting Variables

`CustomerID` is a unique identifier rather than a meaningful behavioral feature. `Count` is a reporting field used for counting customers.

These variables do not provide useful predictive information and will be removed from the modeling dataset.

In [ ]:
cleaned_df.drop(columns = ["CustomerID"],axis = 1,inplace = True)
print(cleaned_df.columns)

Index(['City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender',
       'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months',
       'Phone Service', 'Multiple Lines', 'Internet Service',
       'Online Security', 'Online Backup', 'Device Protection', 'Tech Support',
       'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing',
       'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label',
       'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason'],
      dtype='object')


## Remove Geographic Variables

The dataset contains detailed geographic variables including city, ZIP code, latitude, longitude, and combined geographic coordinates.

These variables are not required for the primary customer behavior-based churn prediction model. They are therefore excluded from the modeling dataset to keep the feature set focused on customer demographics, services, contracts, and billing behavior.

In [ ]:
geographic_columns = [
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude"
]
cleaned_df = cleaned_df.drop(columns=geographic_columns)
print(cleaned_df.columns)

Index(['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months',
       'Phone Service', 'Multiple Lines', 'Internet Service',
       'Online Security', 'Online Backup', 'Device Protection', 'Tech Support',
       'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing',
       'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label',
       'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason'],
      dtype='object')


In [ ]:
print(cleaned_df.shape)

(7043, 24)


## Remove Data Leakage Variables

Several variables contain information that should not be provided to the predictive model.

`Churn Label` directly represents the same outcome as the target variable.

`Churn Score` is an existing predictive score generated by another predictive system.

`Churn Reason` is information that becomes available after a customer has already churned.

Including these variables would lead to data leakage and unrealistic model performance.

Therefore, they are removed from the modeling dataset.

In [ ]:
Data_Leakage_columns = ["Churn Label","Churn Score","Churn Reason"]
cleaned_df = cleaned_df.drop(columns = Data_Leakage_columns)
print(cleaned_df.columns)

Index(['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months',
       'Phone Service', 'Multiple Lines', 'Internet Service',
       'Online Security', 'Online Backup', 'Device Protection', 'Tech Support',
       'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing',
       'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value',
       'CLTV'],
      dtype='object')


In [ ]:
print(cleaned_df.shape)

(7043, 21)


## Target Variable

`Churn Value` is retained because it is the target variable that the machine learning model will learn to predict.

It contains:

- `0` → Customer did not churn
- `1` → Customer churned

In [ ]:
print(cleaned_df["Churn Value"].value_counts())

Churn Value
0    5174
1    1869
Name: count, dtype: int64


In [ ]:
print(cleaned_df.columns.tolist())

['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value', 'CLTV']


In [ ]:
print(cleaned_df.shape,cleaned_df.dtypes)

(7043, 21) Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Phone Service         object
Multiple Lines        object
Internet Service      object
Online Security       object
Online Backup         object
Device Protection     object
Tech Support          object
Streaming TV          object
Streaming Movies      object
Contract              object
Paperless Billing     object
Payment Method        object
Monthly Charges      float64
Total Charges        float64
Churn Value            int64
CLTV                   int64
dtype: object


In [ ]:
#print(cleaned_df.isnull().sum())
print(cleaned_df.duplicated().sum())
print(cleaned_df.shape)

0
(7043, 21)


## Save Cleaned Dataset

The cleaned dataset is saved as a CSV file so that it can be reused in subsequent notebooks without repeating the cleaning process.

In [ ]:
cleaned_df.to_csv("Cleaned_Customer_Churn.csv",index=False
)


In [ ]:
print(cleaned_df.head())
print(cleaned_df.describe)

Streaming output truncated to the last 5000 lines.
2043        4107.25            0  5698  
2044        5760.65            0  6370  
2045        4747.50            0  6367  
2046        1566.90            0  5733  
2047         702.00            0  5491  
2048         299.05            0  3180  
2049        1305.95            0  4283  
2050         284.35            0  4272  
2051        6350.50            0  4530  
2052        7878.30            0  4551  
2053        3187.65            0  5246  
2054        6126.15            0  5348  
2055         731.30            0  5557  
2056         273.40            0  4033  
2057        2531.80            0  5043  
2058        4298.45            0  2887  
2059        4619.55            0  5561  
2060        2633.30            0  3945  
2061         193.05            0  3768  
2062        4103.90            0  4478  
2063        7008.15            0  6090  
2064        5791.10            0  5068  
2065        1228.65            0  3094  
2066  

## Conclusion

In this notebook, the Telco Customer Churn dataset was cleaned and prepared for further analysis and machine learning.

Duplicate records were checked, the `Total Charges` variable was converted to a numeric format, and appropriate missing values were handled.

Constant variables, identifiers, unnecessary geographic variables, and potential data leakage variables were removed from the modeling dataset.

The target variable, `Churn Value`, was retained for binary classification.

The final cleaned dataset contains 7,043 customer records and 21 variables. It is now ready for the next stage of the project: Feature Engineering and Exploratory Data Analysis.